# 12 — GSE225948 longitudinal independent preparations: stroke day 2 versus day 14

Four day-2 stroke and four day-14 stroke preparations; each GEO replicate pools ~4–5 hemispheres. These are **independent pooled preparations, not paired individuals**. There is **no matched sham group at day 14**, so this is descriptive of injury-stage-associated vascular cell expression, not an injury effect relative to normal controls. It requires EC-subtype composition sensitivity. The deposited fractional count matrices are analyzed as source normalized expression rather than integer UMI DESeq2. Two-sided exact label permutation across eight pooled preparations has min p=0.02857. Results do not validate protein/tracer/BBB passage.


In [ ]:
!pip -q install "pandas>=2.1" "numpy>=1.26" "scipy>=1.11" "requests>=2.31" "beautifulsoup4>=4.12" "statsmodels>=0.14"
import gzip, hashlib, itertools, json, os, re
from pathlib import Path
import numpy as np, pandas as pd, requests
from bs4 import BeautifulSoup
from statsmodels.stats.multitest import multipletests

ROOT=Path(os.environ.get("IMACS_ROOT","/content/imacs_stroke_GSE225948" if Path("/content").exists() else "imacs_stroke_GSE225948"))
RAW=ROOT/"raw"; OUT=ROOT/"results"
for x in (RAW,OUT):x.mkdir(exist_ok=True,parents=True)
SESSION=requests.Session()
CAT=pd.read_csv("https://raw.githubusercontent.com/Touf00/retinal-nvu-computational-analysis/imacs-stroke-pet-screen/IMACS_Stroke_PET/config/candidate_catalog.csv").drop_duplicates("gene")
TARGETS=list(CAT.gene)
LINEAGE=["Cldn5","Pecam1","Kdr","Esam","Emcn","Ramp2","Aqp4","Aldh1l1","Slc1a3","Pdgfrb","Rgs5","Acta2","P2ry12","Tmem119","Ptprc","Lyz2","Snap25"]
GENES=set(TARGETS+LINEAGE)
SAMPLES=pd.DataFrame([(f"GSM70608{str(i).zfill(2)}", "Day2" if i<=22 else "Day14")
                      for i in range(19,27)],columns=["gsm","condition"])
assert len(SAMPLES)==8 and set(SAMPLES.condition)=={"Day2","Day14"}
print("Young male tMCAO stroke-only, 4 pooled preparations at day2 vs 4 at day14; not matched by animal")
display(SAMPLES)


## GEO exact filename lookup, metadata/annotation inspection and provenance

Never infer sample membership from clustering or from gene expression. Link the count columns to the paper-provided `metadata.csv` by exact barcode; abort on substantial mismatches.

In [ ]:
def sha256(path):
    d=hashlib.sha256()
    with open(path,"rb") as f:
        for z in iter(lambda:f.read(1<<20),b""):d.update(z)
    return d.hexdigest()

def fetch(url,path):
    if path.exists() and path.stat().st_size>0:return path
    with SESSION.get(url,stream=True,timeout=240) as r:
        r.raise_for_status()
        with open(path,"wb") as f:
            for chunk in r.iter_content(1<<20):
                if chunk:f.write(chunk)
    return path

def source_files(gsm):
    url=f"https://ftp.ncbi.nlm.nih.gov/geo/samples/{gsm[:-3]}nnn/{gsm}/suppl/"
    r=SESSION.get(url,timeout=120);r.raise_for_status()
    links=[a.get("href","") for a in BeautifulSoup(r.text,"html.parser").find_all("a")]
    result={}
    for typ in ("counts","metadata"):
        found=[a for a in links if a.startswith(gsm+"_") and a.endswith("_"+typ+".csv.gz")]
        if len(found)!=1:raise RuntimeError(f"{gsm} {typ} expected exactly one matching file, got {found}")
        result[typ]=url+found[0]
    return result

provenance=[]; all_source={}
for sm in SAMPLES.itertuples(index=False):
    urls=source_files(sm.gsm);all_source[sm.gsm]={}
    for typ,url in urls.items():
        fn=url.rsplit("/",1)[-1]
        p=fetch(url,RAW/fn)
        provenance.append({"gsm":sm.gsm,"condition":sm.condition,"type":typ,"url":url,
                           "file":fn,"bytes":p.stat().st_size,"sha256":sha256(p)})
        all_source[sm.gsm][typ]=p
pd.DataFrame(provenance).to_csv(OUT/"12_inputs_checksums.csv",index=False)
print("Downloaded",len(provenance),"input files")


In [ ]:
META={}
for sm in SAMPLES.itertuples(index=False):
    m=pd.read_csv(all_source[sm.gsm]["metadata"],compression="gzip",index_col=0)
    assert m.index.is_unique,sm.gsm
    required={"parent","sub.celltype","nCount_RNA","nFeature_RNA","treatment","Replicate"}
    assert required.issubset(m.columns),set(m.columns)
    assert m.index.map(str).str.len().min()>5
    META[sm.gsm]=m
    print(sm.gsm,sm.condition,"cells",len(m),"parent:",m["parent"].value_counts().head(15).to_dict())
    assert (m["nCount_RNA"]>0).all(),sm.gsm
annotation=[]
for sm in SAMPLES.itertuples(index=False):
    m=META[sm.gsm]
    for ct,n in m["parent"].value_counts().items():
        annotation.append({"gsm":sm.gsm,"condition":sm.condition,"author_parent":ct,"n_cells":int(n)})
pd.DataFrame(annotation).to_csv(OUT/"12_author_annotated_population_counts.csv",index=False)


## Gene-panel extraction without loading whole gene-by-cell matrices into RAM

Each count matrix has genes in rows and cell barcodes in columns. The published matrices have fractional values (first source row verified), so we retain them as such. Stream 256 gene rows at a time and retrieve only frozen genes and lineage sentinels, without performing de novo clustering on this already FACS-enriched dataset.

In [ ]:
EXPR={}; file_audits=[]
for sm in SAMPLES.itertuples(index=False):
    path=all_source[sm.gsm]["counts"]
    selected=[]
    for chunk in pd.read_csv(path,compression="gzip",index_col=0,chunksize=256):
        keep=chunk.index.astype(str).isin(GENES)
        if keep.any():selected.append(chunk.loc[keep])
    if not selected:raise RuntimeError(f"None of prespecified genes found in {sm.gsm}")
    e=pd.concat(selected,axis=0)
    duplicate=int(e.index.duplicated().sum())
    if duplicate:e=e.groupby(level=0).sum()
    e.index=e.index.astype(str)
    e.columns=e.columns.astype(str)
    meta=META[sm.gsm]; matching=e.columns.intersection(meta.index.astype(str))
    unmatched=len(e.columns)-len(matching)
    if len(matching)<0.98*len(e.columns):raise RuntimeError(f"{sm.gsm} unmatched metadata barcodes {unmatched}/{len(e.columns)}")
    if (e.to_numpy()<0).any():raise RuntimeError(f"{sm.gsm} has negative expression values")
    nonint=float((np.abs(e.to_numpy()-np.rint(e.to_numpy()))>1e-7).mean())
    EXPR[sm.gsm]=e.loc[:,matching]
    file_audits.append({"gsm":sm.gsm,"cells_expression":len(e.columns),"matched_metadata":len(matching),
                        "unmatched":unmatched,"panel_genes":len(e),"duplicate_feature_names":duplicate,
                        "fractional_value_fraction":nonint})
pd.DataFrame(file_audits).to_csv(OUT/"12_matrix_format_and_barcode_QC.csv",index=False)
display(pd.DataFrame(file_audits))


## Independent-preparation disease-only time contrast

Use author-supplied parent/sub.celltype annotations. Mark EC exactly by the original `parent` label(s), not candidate-driven markers. Fail if a preparation has fewer than 10 annotated ECs. This is **not** cell-level DE, and any significant result only refers to the preparation-level sampled population.

In [ ]:
alltypes=sorted(set(str(x) for m in META.values() for x in m["parent"].unique()))
print("Original parent labels:",alltypes)
EC_LABELS=[x for x in alltypes if re.fullmatch(r"EC|ECs|Endothelial|EndothelialCells|Endothelium",x,flags=re.I)]
assert EC_LABELS, f"No unambiguous author-provided EC label in {alltypes}; inspect metadata."
print("Exact EC labels:",EC_LABELS)
summary=[]
for sm in SAMPLES.itertuples(index=False):
    m=META[sm.gsm]
    e=EXPR[sm.gsm]
    ec_ids=m.index[m["parent"].astype(str).isin(EC_LABELS)].astype(str).intersection(e.columns)
    if len(ec_ids)<10:
        raise RuntimeError(f"Only {len(ec_ids)} endothelial cells in {sm.gsm}; do not run endothelial inference")
    raw=e.loc[:,ec_ids]
    lib=m.loc[ec_ids,"nCount_RNA"].astype(float)
    nc=raw.div(lib,axis=1)*1e4
    normalized=np.log1p(nc)
    for gene in TARGETS:
        if gene in e.index:
            x=raw.loc[gene].to_numpy(dtype=float)
            y=normalized.loc[gene].to_numpy(dtype=float)
            summary.append({"gsm":sm.gsm,"condition":sm.condition,"gene":gene,
                            "status":"TESTABLE","n_author_EC_cells":len(ec_ids),"mean_cell_log1p_10000":float(y.mean()),
                            "pct_cells_detected":float((x>0).mean()),
                            "aggregate_panel_gene_counts":float(x.sum()),
                            "mean_fractional_counts":float(x.mean())})
        else:
            summary.append({"gsm":sm.gsm,"condition":sm.condition,"gene":gene,
                            "status":"NOT_MATCHED_TO_SOURCE_FEATURE","n_author_EC_cells":len(ec_ids),
                            "mean_cell_log1p_10000":np.nan,"pct_cells_detected":np.nan})
T=pd.DataFrame(summary)
T.to_csv(OUT/"12_day2_EC_gene_by_preparation.csv",index=False)
display(T.pivot_table(index="gene",columns="condition",values="mean_cell_log1p_10000",aggfunc="mean"))


In [ ]:
result=[]
for gene, grp in T[T.status=="TESTABLE"].groupby("gene"):
    v=grp.set_index("gsm").loc[SAMPLES.gsm]
    x=v["mean_cell_log1p_10000"].to_numpy(float)
    cond=SAMPLES.condition.eq("Day14").to_numpy()
    assert sum(cond)==4 and len(x)==8 and not np.isnan(x).any()
    delta=float(x[cond].mean()-x[~cond].mean())
    perms=[]
    for combination in itertools.combinations(range(8),4):
        k=np.zeros(8,dtype=bool);k[list(combination)]=True
        perms.append(float(x[k].mean()-x[~k].mean()))
    p=float(np.mean(np.abs(perms)>=abs(delta)-1e-12))
    result.append({"gene":gene,"n_day2_stroke_preparations":4,"n_day14_stroke_preparations":4,
                   "day14_minus_day2_mean_log1p_10000":delta,"exact_two_sided_p":p,
                   "all_4_day14_above_day2":bool(x[cond].min()>x[~cond].max()),
                   "all_4_day14_below_day2":bool(x[cond].max()<x[~cond].min()),
                   "min_cells_in_prep":int(v["n_author_EC_cells"].min()),
                   "n_individual_animals":"UNKNOWN: source pools 4–5 hemispheres per experiment"})
R=pd.DataFrame(result)
if len(R):R["BH_q_across_frozen_panel"]=multipletests(R["exact_two_sided_p"],method="fdr_bh")[1]
R.to_csv(OUT/"12_GSE225948_day2_EC_exact_permutation.csv",index=False)
print("Contrasts from independent pooled PREPARATIONS, NOT individual animals")
display(R.sort_values(["exact_two_sided_p","gene"]))


In [ ]:

## Within-source-endothelial subtype audit; inferential contrast only if subtype is captured in >=30 cells in all eight pools.
# EC2 is often scarcely sampled in Sham (as few as one cell), so no EC2 inferential contrast.
SUBTYPES=["EC1","EC2"]
subrows=[]
for sm in SAMPLES.itertuples(index=False):
    m=META[sm.gsm]
    e=EXPR[sm.gsm]
    for subtype in SUBTYPES:
        ids=m.index[m["sub.celltype"].astype(str)==subtype].astype(str).intersection(e.columns)
        if len(ids)==0:continue
        nr=m.loc[ids,"nCount_RNA"].to_numpy(float)
        for gene in TARGETS:
            if gene not in e.index:continue
            x=e.loc[gene,ids].to_numpy(dtype=float)
            y=np.log1p(x/np.maximum(nr,1)*1e4)
            subrows.append({"gsm":sm.gsm,"condition":sm.condition,"author_subtype":subtype,
                           "gene":gene,"n_cells":len(ids),"mean_log1p_10000":float(y.mean()),
                           "pct_detected":float((x>0).mean())})
S=pd.DataFrame(subrows)
S.to_csv(OUT/"12_author_EC1_EC2_sample_gene_descriptive.csv",index=False)
MIN_PER_PREP=30
subresults=[]
for (subtype,gene),g in S.groupby(["author_subtype","gene"]):
    if g.gsm.nunique()!=8 or int(g.n_cells.min())<MIN_PER_PREP:
        subresults.append({"author_subtype":subtype,"gene":gene,"status":"NO_INFERENCE_low_or_missing_group_sample"})
        continue
    q=g.set_index("gsm").loc[SAMPLES.gsm]
    x=q["mean_log1p_10000"].to_numpy(float)
    is_case=SAMPLES.condition.eq("Day14").to_numpy()
    delta=float(x[is_case].mean()-x[~is_case].mean())
    p=float(np.mean([abs(float(x[np.array(list(c))].mean()-x[[i for i in range(8) if i not in c]].mean()))>=abs(delta)-1e-12
                     for c in itertools.combinations(range(8),4)]))
    subresults.append({"author_subtype":subtype,"gene":gene,"status":"TESTABLE_4v4_pooled_preps",
                       "delta_aggregate_mean_log1p_10000":delta,"exact_two_sided_p":p,
                       "min_n_cells_in_preparation":int(q.n_cells.min())})
Z=pd.DataFrame(subresults)
Z["BH_q_within_subtype_frozen_panel"]=np.nan
for st in SUBTYPES:
    flag=(Z.author_subtype==st)&(Z.status=="TESTABLE_4v4_pooled_preps")
    if flag.sum()>0:
        Z.loc[flag,"BH_q_within_subtype_frozen_panel"]=multipletests(Z.loc[flag,"exact_two_sided_p"],method="fdr_bh")[1]
Z.to_csv(OUT/"12_within_author_EC_subtype_sensitivity.csv",index=False)
display(Z.sort_values(["author_subtype","exact_two_sided_p"],na_position="last").head(50))
print("Subtype selection is post-hoc on the original author clusters and cannot establish cell lineage or causal phenotypic transition.")


## Final audit and boundary

- Each sample represents a **pooled preparation**; group-label permutation inference requires exchangeability of preparations, not individual mice.
- Whole-brain dissociation and FACS sorting affect sampled cell fractions.
- Stored matrices contain fractional expression and **must not** be described as original raw integer UMI counts.
- No claim about PET tracers follows from a gene's RNA direction.
- Day-14 stroke-only preparations have **no matched sham day-14 comparator** and were excluded.
- All p-values relate only to the prespecified panel and were adjusted only across that panel.


In [ ]:
(OUT/"12_run_manifest.json").write_text(json.dumps({
"dataset":"GSE225948","cells":"original author EC parent label","contrast":"young-male tMCAO stroke brain day14 vs day2, independent pooled preparations; NO SHAM day14",
"independent_units":"n=4 pooled preparations/group, 4–5 animal hemispheres per preparation",
"statistics":"exact label permutation on preparation-level mean log1p expression; BH 29-gene panel",
"source_matrix_unit":"fractional-valued deposited counts.csv, not integer UMI"
},indent=2))
print("DONE",len(list(OUT.glob("*"))),"outputs")
